In [0]:
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth

APP_TOKEN = ""
EMAIL = ""
API_SECRET = ""

url = "https://data.cityofchicago.org/resource/ajtu-isnz.json"
chunk_size = 100_000

# params = {"$limit": chunk_size, "$offset": offset


headers = {"X-App-Token": APP_TOKEN}
auth = HTTPBasicAuth(EMAIL, API_SECRET)

resp = requests.get(
    url,
    params={"$select": "min(trip_start_timestamp) as min_date, max(trip_start_timestamp) as max_date"},
    headers=headers,
    auth=auth
)

print(resp.json())

        


In [0]:
from datetime import datetime, timedelta

In [0]:
start_date = datetime.strptime(resp.json()[0]["min_date"].split('T')[0], "%Y-%m-%d").strftime("%Y-%m-%d")
end_date = datetime.strptime(resp.json()[0]["max_date"].split('T')[0], "%Y-%m-%d").strftime("%Y-%m-%d")

In [0]:
import pandas as pd

dates = pd.date_range(start=start_date, end=end_date, freq="MS")

date_ranges = [
    (str(dates[i].date()), str(dates[i+1].date()))
    for i in range(len(dates) - 1)
]

print(date_ranges)

In [0]:
def split_range(start_date, end_date, next_level):
    start = datetime.fromisoformat(start_date)
    end = datetime.fromisoformat(end_date)

    delta = 7 if next_level == "week" else 1

    while start < end:
        next_step = min(start + timedelta(days=delta), end)

        fetch_range(
            start.strftime("%Y-%m-%d"),
            next_step.strftime("%Y-%m-%d"),
            level=next_level
        )

        start = next_step

In [0]:
import requests
import time
from datetime import datetime, timedelta

LIMIT = 500000

buffer = []
BUFFER_LIMIT = 500000  # tune this

def fetch_range(start_date, end_date, level="month"):

    params = {
        "$select": "trip_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude",
        "$where": f"trip_start_timestamp >= '{start_date}T00:00:00' AND trip_start_timestamp < '{end_date}T00:00:00'",
        "$limit": LIMIT
    }

    for attempt in range(3):
        try:
            response = requests.get(
                url,
                headers=headers,
                auth=auth,
                params=params,
                timeout=60
            )

            response.raise_for_status()
            data = response.json()

            if not data:
                print(f"⚪ No data {start_date} → {end_date}")
                return

            # KEY LOGIC: detect truncation
            if len(data) == LIMIT:
                print(f"Too large → splitting {start_date} → {end_date}")

                if level == "month":
                    split_range(start_date, end_date, "week")
                elif level == "week":
                    split_range(start_date, end_date, "day")
                else:
                    print(f"Still too large at day level: {start_date} → {end_date}")

                return  # do NOT write truncated data

            # ✅ Safe to write
            buffer.extend(data)

            if len(buffer) >= BUFFER_LIMIT:
                df = spark.createDataFrame(buffer)
                
                df = df.dropDuplicates(["trip_id"])

                df.write.format("delta") \
                    .mode("append") \
                    .save("abfss://raw@datalakeimes.dfs.core.windows.net/taxi_trips")

                print(f"Written batch: {len(buffer)} rows")

                buffer.clear()

            print(f"{start_date} → {end_date} | rows: {len(data)}")

            
            return

        except Exception as e:
            print(f"Retry {attempt+1} failed: {start_date} → {end_date} | {e}")
            time.sleep(2 ** attempt)

    # retries failed → split anyway
    print(f"🔀 Retry failed → splitting {start_date} → {end_date}")

    if level == "month":
        split_range(start_date, end_date, "week")
    elif level == "week":
        split_range(start_date, end_date, "day")
    else:
        print(f"Failed even at day level: {start_date} → {end_date}")

In [0]:
# NO threading
for start, end in date_ranges:
    fetch_range(start, end)

In [0]:
if buffer:
    df = spark.createDataFrame(buffer)
    df = df.dropDuplicates(["trip_id"])

    df.write.format("delta") \
        .mode("append") \
        .save("abfss://raw@datalakeimes.dfs.core.windows.net/taxi_trips")

    print(f"Final write: {len(buffer)} rows")

In [0]:
%sql
-- SELECT COUNT(*) FROM bronze.trips.trips_table